# EchoPilot Research 04: Bayesian Knowledge Tracing (BKT) vs. Deep Knowledge Tracing (DKT)

In the current EchoPilot web app, mastery is updated using a simple moving average:
$$S_{t+1} = 0.72 \cdot S_t + 0.28 \cdot (100 \text{ if correct else } 0)$$

In this research notebook, we implement **Bayesian Knowledge Tracing (BKT)**:
- Hidden Markov Model tracking latent mastery state $L_t \in \{0, 1\}$.
- Parameters: Prior $P(L_0)$, Transition $P(T)$, Guess $P(G)$, and Slip $P(S)$.
- Parameter fitting via Baum-Welch Expectation-Maximization (EM).
- Comparative evaluation against a **Deep Knowledge Tracing (DKT with LSTM)** PyTorch neural baseline.
- Metrics: AUC-ROC, Brier Score, Log Loss, and Calibration Error (ECE).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))
from bkt import BKTModel
from dkt import DKTNet, encode_interactions

print("BKT and DKT modules loaded.")


## Step 1: Load Empirical Interaction Sequences
We load the student response sequences from `../data/sample_interactions.csv`.


In [ ]:
df_interactions = pd.read_csv("../data/sample_interactions.csv")
print(f"Loaded {len(df_interactions)} interaction records.")
print(df_interactions.head(6))


## Step 2: Fit BKT via Baum-Welch Expectation-Maximization (EM)
We group interaction logs by student and skill, and fit the 4 BKT parameters per skill.


In [ ]:
fitted_bkt_models = {}

for skill in df_interactions["skill_name"].unique():
    skill_df = df_interactions[df_interactions["skill_name"] == skill]
    
    # Extract binary response sequences per student
    student_seqs = [group["is_correct"].tolist() for _, group in skill_df.groupby("student_id")]
    
    bkt = BKTModel()
    fitted_params = bkt.fit_em(student_seqs, max_iter=30)
    fitted_bkt_models[skill] = bkt
    
    metrics = bkt.evaluate_metrics(student_seqs)
    print(f"\nSkill: '{skill}'")
    print(f"  Fitted Parameters: {fitted_params}")
    print(f"  AUC-ROC:           {metrics.get('auc_roc')}")
    print(f"  Brier Score:       {metrics.get('brier_score')}")
    print(f"  Log Loss:          {metrics.get('log_loss')}")
    print(f"  ECE (Calibration): {metrics.get('ece')}")


## Step 3: Forward Bayesian Mastery Update Simulation
Demonstrate how a student's estimated mastery updates question-by-question for a sequence of 6 answers:
`[Incorrect (0), Correct (1), Correct (1), Correct (1), Incorrect (0), Correct (1)]`.


In [ ]:
test_skill = list(fitted_bkt_models.keys())[0]
model = fitted_bkt_models[test_skill]

sample_answers = [0, 1, 1, 1, 0, 1]
preds, masteries = model.forward_sequence(sample_answers)

trace_df = pd.DataFrame({
    "Opportunity": range(1, len(sample_answers) + 1),
    "Answer": ["Incorrect", "Correct", "Correct", "Correct", "Incorrect", "Correct"],
    "Pred_P(Correct)": [round(p, 3) for p in preds],
    "Updated_Mastery": [round(m, 3) for m in masteries]
})

print(f"BKT Forward Belief Progression for '{test_skill}':")
print(trace_df.to_string(index=False))


## Step 4: Deep Knowledge Tracing (DKT) PyTorch Baseline
We train an LSTM-based Deep Knowledge Tracing network on the multi-skill dataset for comparison.


In [ ]:
skills_list = list(df_interactions["skill_name"].unique())
skill2idx = {s: i for i, s in enumerate(skills_list)}
num_skills = len(skills_list)

# Encode sequences
sequences = []
for student_id, group in df_interactions.groupby("student_id"):
    seq = [(skill2idx[row["skill_name"]], int(row["is_correct"])) for _, row in group.iterrows()]
    if len(seq) >= 4:
        sequences.append(seq)

print(f"Prepared {len(sequences)} student trajectories for DKT.")

# Simple DKT training demonstration
dkt = DKTNet(num_skills=num_skills, hidden_dim=32, num_layers=1)
optimizer = torch.optim.Adam(dkt.parameters(), lr=0.01)
criterion = torch.nn.BCELoss()

dkt.train()
losses = []
for epoch in range(15):
    epoch_loss = 0.0
    for seq in sequences:
        x, target_skills, target_correct = encode_interactions(seq, num_skills)
        x_t = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
        y_skills = torch.tensor(target_skills, dtype=torch.int64)
        y_corr = torch.tensor(target_correct, dtype=torch.float32)
        
        preds = dkt(x_t).squeeze(0)  # [seq_len, num_skills]
        # Gather predictions for the specific skill targeted at each step
        skill_preds = preds.gather(1, y_skills.unsqueeze(1)).squeeze(1)
        
        loss = criterion(skill_preds, y_corr)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    losses.append(epoch_loss / len(sequences))

print(f"DKT Training completed. Initial loss: {losses[0]:.4f} -> Final loss: {losses[-1]:.4f}")


## Step 5: Research Conclusion & Selection Recommendation
- **Interpretability**: BKT parameters ($P(L_0), P(T), P(G), P(S)$) map directly to educational psychometrics (prior knowledge, learning rate, guessing, slipping). DKT weights are uninterpretable black-box tensors.
- **Data Efficiency**: BKT fits reliably on small datasets (50-100 sequences), whereas DKT requires thousands of sequences to avoid overfitting.
- **Explainability**: EchoPilot can tell the student: *"We noticed a 35% probability of mastery on 'Circular Wait' based on your recent 2 attempts, so we scheduled targeted practice."*
- **Verdict**: Keep **BKT** as EchoPilot's primary production model.
